# **Import Libraries**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# **1- Data Gathering (Kaggel)**

In [2]:
df = pd.read_csv("/content/train.txt", sep = ";", header = None, names = ["text", "emotion"])

# **2- Text Cleaning**

In [3]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [4]:
df.isnull().sum()

,0
text,0
emotion,1


In [5]:
unique_emotions = df["emotion"].unique()
emotion_numbers = {}
i = 0
for emo in unique_emotions:
  emotion_numbers[emo] = i
  i += 1

df["emotion"] = df["emotion"].map(emotion_numbers)

In [6]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


**2.1- Lowercasing**

In [7]:
df["text"] = df["text"].apply(lambda x : x.lower())

**2.2- Remove Punctuation**

In [8]:
import string

def remove_punc(txt):
  return txt.translate(str.maketrans("","",string.punctuation))

In [9]:
df["text"] = df["text"].apply(remove_punc)

**2.3- Remove Numbers**

In [10]:
def remove_num(txt):
  new = ""
  for i in txt:
    if not i.isdigit():
      new = new + i
  return new

df["text"] = df["text"].apply(remove_num)

**2.4- Remove URLs/Links**

In [11]:
import re

def remove_urls(txt):
    return re.sub(r'https?://\S+|www\.\S+', '', txt)

df["text"] = df["text"].apply(remove_urls)

**2.5- Remove Emojis**

In [12]:
def remove_emojis(txt):

    new = ""

    for i in txt:
        if i.isascii():
            new = new + i

    return new

df["text"] = df["text"].apply(remove_emojis)

In [13]:
df.loc[1]["text"]

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

**2.6- Remove Stopwards**

In [14]:
import nltk

In [15]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [16]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [17]:
stop_words = set(stopwords.words('english'))
stop_words

{'a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 "he's",
 'her',
 'here',
 'hers',
 'herself',
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 "i'll",
 "i'm",
 "i've",
 'if',
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [18]:
len(stop_words)

198

In [19]:
def remove(txt):
  words = txt.split()
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)

  return " ".join(cleaned)

In [20]:
df["text"] = df["text"].apply(remove)

In [21]:
df.loc[1]["text"]

'go feeling hopeless damned hopeful around someone cares awake'

# **3- Train-Test Split**

In [23]:
from sklearn.model_selection import train_test_split

In [25]:
X_train, X_test, y_train, y_test = train_test_split(df["text"], df["emotion"], test_size=0.33, random_state=42)

# **4- Vectorization**

**4.1- Bag of Words**

In [53]:
from sklearn.feature_extraction.text import CountVectorizer

In [54]:
bow_vectorizer = CountVectorizer()

In [55]:
X_train_bow = bow_vectorizer.fit_transform(X_train)

In [56]:
X_train_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 61531 stored elements and shape (6735, 9482)>

In [57]:
X_test_bow = bow_vectorizer.transform(X_test)

In [58]:
X_test_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 27446 stored elements and shape (3318, 9482)>

**Model Training**

**Naive Bayes**

In [59]:
from sklearn.naive_bayes import MultinomialNB

In [60]:
from sklearn.metrics import accuracy_score

In [61]:
nb_model = MultinomialNB()

In [62]:
nb_model.fit(X_train_bow, y_train)

MultinomialNB()

In [63]:
pred_nb = nb_model.predict(X_test_bow)

In [65]:
accuracy_score(y_test, pred_nb)

0.7284508740204942

**4.2- TF-IDF**

In [66]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [68]:
tfidf = TfidfVectorizer()

In [69]:
X_train_tfidf = tfidf.fit_transform(X_train)

In [70]:
X_test_tfidf = tfidf.transform(X_test)

**Model Training**

**Naive Bayes**

In [71]:
from sklearn.naive_bayes import MultinomialNB

In [72]:
nb_model = MultinomialNB()

In [73]:
nb_model.fit(X_train_tfidf, y_train)

MultinomialNB()

In [74]:
pred_nb = nb_model.predict(X_test_tfidf)

In [75]:
accuracy_score(y_test, pred_nb)

0.6341169379144063

**Model Training**

**Logistic Regression**

In [76]:
from sklearn.linear_model import LogisticRegression

In [77]:
lr_model = LogisticRegression(max_iter=1000)

In [78]:
lr_model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [80]:
pred_lr = lr_model.predict(X_test_tfidf)

In [81]:
accuracy_score(y_test, pred_lr)

0.8119349005424955